# cassette — record the call once, replay it forever

Every test run that hits a real model costs money and flakes. `cassette` records the exchange the first time and replays it afterwards: same assertion, **zero** calls, no network.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · Record, then replay

`mode="auto"` decides by whether the file exists — which is what makes a cassette usable in CI without a flag.

In [ ]:
import pathlib
import tempfile

import main as recipe

with tempfile.TemporaryDirectory() as d:
    r = recipe._record_then_replay(str(pathlib.Path(d) / "run.json"))
rec, rep = r["record"], r["replay"]
rec, rep

## 2 · The call count is the claim

A replay recipe that compared only the OUTPUT would pass just as happily if the "replay" quietly re-called the provider.

In [ ]:
print(f"run 1: recorded ({rec['n']} call, {rec['ms']:.1f} ms)")
print(f"run 2: replayed ({rep['n']} calls, offline, {rep['ms']:.1f} ms)")

## 3 · Prove it

In [ ]:
assert rec["n"] == 1, "the first run should have made exactly one real call"
assert rep["n"] == 0, "the replay reached the client — it is not offline"
assert rec["out"] == rep["out"]
print("OK")